# Scan-count ablation

Binary development dataset: 130 samples, 32 raw scans each. Same experiment as the verified Python script.

Run cells from top to bottom if you want to reproduce the analysis. Your completed results do not need to be rerun. Physical timing is recorded separately.

## 1. Import libraries

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


## 2. Set the experiment settings

Keep these values unchanged to reproduce the completed experiment. The subsampling seed controls scan selection; it is separate from the model seed.

In [ ]:
FEATURES = ['730nm', '760nm', '810nm', '860nm', '900nm', '940nm']
LABELS = {'Not Sweet': 0, 'Sweet': 1}
COUNTS = [1, 5, 10, 15, 20, 32]
REPETITIONS = 30
SUBSAMPLING_SEED = 20260909


## 3. Set the XGBoost parameters

Full-precision values from the verified model.

In [ ]:
PARAMS = {
    'n_estimators': 135,
    'max_depth': 2,
    'learning_rate': 0.04772753454707194,
    'min_child_weight': 5.708054401119242,
    'gamma': 0.29256771005742016,
    'subsample': 0.9879709827503215,
    'colsample_bytree': 0.9936378122095748,
    'reg_lambda': 1.4992089279125758,
    'reg_alpha': 0.07944657116549703,
    'max_delta_step': 2,
    'objective': 'binary:logistic',
    'random_state': None
}


## 4. Keep the original sample partitions

These are sample IDs, not raw scan row numbers. Fitting uses training + validation; testing uses only the 20 test samples.

In [ ]:
TRAIN_IDS = [70, 42, 120, 16, 4, 119, 76, 53, 39, 116, 59, 62, 25, 12, 72, 24, 31, 80, 98, 45,
 73, 28, 46, 107, 104, 6, 85, 5, 87, 47, 122, 71, 30, 10, 22, 81, 111, 44, 94, 34,
 26, 112, 18, 35, 92, 50, 2, 91, 86, 61, 68, 88, 60, 51, 65, 128, 114, 125, 124, 79,
 11, 109, 49, 90, 127, 66, 97, 108, 32, 63, 117, 8, 69, 14, 83, 33, 121, 96, 41, 126,
 37, 113, 17, 115, 40, 1, 84, 75, 9, 118, 64]


In [ ]:
VAL_IDS = [19, 106, 36, 7, 56, 89, 23, 48, 78, 123, 54, 15, 100, 20, 58, 103, 21, 55, 82]


In [ ]:
TEST_IDS = [99, 67, 43, 93, 29, 27, 52, 74, 129, 105, 77, 13, 57, 95, 102, 101, 38, 130, 3, 110]


## 5. Load and check the raw dataset

In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


In [ ]:
def read_data(path):
    df = pd.read_csv(path)
    required = FEATURES + ['Sample No.', 'Class', 'Brix (%)']
    require(set(required).issubset(df.columns), 'Missing required dataset columns.')
    df = df[required].copy()
    require(not df.isna().any().any(), 'Dataset has missing values.')
    require(np.isfinite(df[FEATURES + ['Brix (%)']].to_numpy(float)).all(),
            'Dataset contains nonfinite numeric values.')
    require(set(df['Sample No.']) == set(range(1, 131)), 'Expected sample IDs 1-130.')
    g = df.groupby('Sample No.', sort=True)
    require(len(df) == 4160 and g.size().eq(32).all(), 'Expected 32 scans for each of 130 samples.')
    require(g[['Class', 'Brix (%)']].nunique().eq(1).all().all(),
            'A sample has inconsistent reference labels or Brix values.')
    samples = g.first()
    expected = np.where(samples['Brix (%)'] >= 8.0, 'Sweet', 'Not Sweet')
    require((samples['Class'] == expected).all(), 'Labels do not match the 8.0 Brix cutoff.')
    require(samples['Class'].value_counts().to_dict() == {'Not Sweet': 67, 'Sweet': 63},
            'Sample class counts differ from the verified dataset.')
    # Reconstruct and compare ordered IDs, not just partition sizes.
    ids = samples.index.to_numpy()
    y = samples['Class'].map(LABELS).to_numpy()
    train, temp, _, ytemp = train_test_split(ids, y, test_size=.3, stratify=y, random_state=42)
    val, test = train_test_split(temp, test_size=.5, stratify=ytemp, random_state=42)
    require(train.tolist() == TRAIN_IDS and val.tolist() == VAL_IDS and test.tolist() == TEST_IDS,
            'Reconstructed partitions differ from the verified original split.')
    # Scan position is within sample in CSV order; it is not a sensor timestamp.
    df['_scan_number'] = g.cumcount() + 1
    return df


In [ ]:
data_path = Path('../datasets/binary/raw_dataset.csv')
df_raw = read_data(data_path)
print('Raw scans:', len(df_raw))
print('Samples:', df_raw['Sample No.'].nunique())
df_raw.head()


## 6. Select scans

Each sample gets a reproducible random scan order per repetition. Taking its first k scans makes subsets nested: 1 is included in 5, 5 in 10, and so on. Original row order is restored before preprocessing.

In [ ]:
def select_scans(df, k, repetition):
    parts = []
    for sid, group in df.groupby('Sample No.', sort=True):
        if k == 32:
            chosen = np.arange(32)
        else:
            seed = np.random.SeedSequence([SUBSAMPLING_SEED, repetition, int(sid)])
            rng = np.random.Generator(np.random.PCG64(seed))
            chosen = np.sort(rng.permutation(32)[:k])
        parts.append(group.iloc[chosen])
    return pd.concat(parts, ignore_index=True)


## 7. Filter outlier scans

In [ ]:
def filter_scans_stage1(df, keep_ratio_fallback=0.85):
    filtered_parts = []

    for sample_id, group in df.groupby('Sample No.'):
        X = group[FEATURES].values
        centroid = X.mean(axis=0)
        dists = np.linalg.norm(X - centroid, axis=1)

        med = np.median(dists)
        mad = np.median(np.abs(dists - med))
        mask = dists <= med + 1.5 * mad

        if mask.sum() < len(group) * 0.5:
            cutoff = np.quantile(dists, keep_ratio_fallback)
            mask = dists <= cutoff

        filtered_parts.append(group[mask])

    return pd.concat(filtered_parts, ignore_index=True)


## 8. Apply SNV

SNV normalizes the six wavelengths within each scan. It also works when only one scan is selected.

In [ ]:
def snv_row(row):
    row = row.astype(float)
    mean = row.mean()
    std = row.std(ddof=1)

    if std < 1e-8:
        return row * np.nan

    return (row - mean) / std


## 9. Combine preprocessing and median aggregation

The result has one six-feature vector per sample. We also return retained scans for the saved records.

In [ ]:
def preprocess(selected):
    filtered = filter_scans_stage1(selected)

    snv = filtered.copy()
    snv[FEATURES] = snv[FEATURES].apply(snv_row, axis=1, result_type='expand')
    snv = snv.dropna()

    vectors = snv.groupby('Sample No.')[FEATURES].median()

    require(set(vectors.index) == set(range(1, 131)),
            'Preprocessing lost a sample. Stop and check the data.')
    require(np.isfinite(vectors.to_numpy()).all(),
            'An aggregated feature is not finite.')

    return vectors, filtered, snv


## 10. Train and evaluate one run

A fresh model is fitted each time. The function returns scores, test predictions, scan records, and the fitted model. The saved deployment model is never loaded.

In [ ]:
def evaluate(df, k, repetition):
    selected = select_scans(df, k, repetition)
    vectors, filtered, snv = preprocess(selected)
    reference = df.groupby('Sample No.')['Class'].first().map(LABELS)
    fit_ids = TRAIN_IDS + VAL_IDS  # Preserve original concatenation order.
    model = XGBClassifier(**PARAMS)
    model.fit(vectors.loc[fit_ids].to_numpy(), reference.loc[fit_ids].to_numpy())
    ytest = reference.loc[TEST_IDS].to_numpy()
    pred = model.predict(vectors.loc[TEST_IDS].to_numpy())
    cm = confusion_matrix(ytest, pred, labels=[0, 1])
    metrics = {
        'scan_count': k, 'repetition': repetition,
        'test_accuracy': float(accuracy_score(ytest, pred)),
        'test_macro_f1': float(f1_score(ytest, pred, labels=[0, 1], average='macro', zero_division=0)),
        'not_sweet_correct': int(cm[0, 0]), 'not_sweet_as_sweet': int(cm[0, 1]),
        'sweet_as_not_sweet': int(cm[1, 0]), 'sweet_correct': int(cm[1, 1]),
        'raw_scan_total': len(selected), 'stage1_scan_total': len(filtered),
        'snv_scan_total': len(snv), 'fit_samples': 110, 'test_samples': 20,
    }
    predictions = [dict(scan_count=k, repetition=repetition, sample_id=sid,
                        reference_label=int(actual), predicted_label=int(guess))
                   for sid, actual, guess in zip(TEST_IDS, ytest, pred)]
    kept = filtered.groupby('Sample No.')['_scan_number'].apply(list)
    valid = snv.groupby('Sample No.')['_scan_number'].apply(list)
    selections = []
    for sid, group in selected.groupby('Sample No.'):
        selections.append(dict(
            scan_count=k, repetition=repetition, sample_id=int(sid),
            selected_scan_numbers=';'.join(map(str, group['_scan_number'])),
            stage1_scan_numbers=';'.join(map(str, kept.get(sid, []))),
            snv_scan_numbers=';'.join(map(str, valid.get(sid, []))),
        ))
    return metrics, predictions, selections, model


## 11. Check the 32-scan baseline

Expected: accuracy 0.85 and macro-F1 0.846547. A mismatch stops the analysis for review; do not retune using the test results.

In [ ]:
baseline = evaluate(df_raw, 32, 0)
baseline_metrics = baseline[0]
print('Accuracy:', baseline_metrics['test_accuracy'])
print('Macro-F1:', baseline_metrics['test_macro_f1'])

matrix_values = [baseline_metrics[name] for name in [
    'not_sweet_correct', 'not_sweet_as_sweet',
    'sweet_as_not_sweet', 'sweet_correct'
]]
print('Confusion matrix:')
print(np.array(matrix_values).reshape(2, 2))
require(matrix_values == [10, 0, 3, 7], 'The baseline differs from the verified result.')


## 12. Prepare the results folder and records

A new folder prevents overwriting previous results. Scores and detailed records are saved after each run.

In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
output_folder = Path('scan_ablation_results_' + stamp)
output_folder.mkdir(exist_ok=False)

records = []

sample_info = df_raw.groupby('Sample No.').first()
partition_rows = []
for name, ids in [('train', TRAIN_IDS), ('validation', VAL_IDS), ('test', TEST_IDS)]:
    for position, sample_id in enumerate(ids, 1):
        partition_rows.append({
            'sample_id': sample_id,
            'partition': name,
            'partition_order': position,
            'reference_label': sample_info.loc[sample_id, 'Class'],
            'brix': sample_info.loc[sample_id, 'Brix (%)']
        })

pd.DataFrame(partition_rows).to_csv(output_folder / 'sample_partitions.csv', index=False)


In [ ]:
metadata = {
    'status': 'running',
    'started_utc': stamp,
    'dataset_sha256': hashlib.sha256(data_path.read_bytes()).hexdigest(),
    'versions': {
        'python': platform.python_version(), 'numpy': np.__version__,
        'pandas': pd.__version__, 'sklearn': sklearn.__version__,
        'xgboost': xgboost.__version__
    },
    'parameters': PARAMS,
    'label_encoding': LABELS,
    'split_seed': 42,
    'subsampling_seed': SUBSAMPLING_SEED,
    'repetitions': REPETITIONS,
    'sampling': 'Nested PCG64 subsets, SeedSequence([subsampling_seed, repetition, sample_id])',
    'sd_definition': 'Sample SD (ddof=1) across subsampling repetitions; N/A for 32 scans.',
    'measurement_time': 'Measured separately on the prototype.',
    'fit_partition': 'Original training + validation: 110 samples; test: 20 samples.',
    'source_format': 'Notebook conversion of the verified scan_count_ablation.py',
    'baseline_booster_config': json.loads(baseline[3].get_booster().save_config())
}

with open(output_folder / 'run_metadata.json', 'w') as file:
    json.dump(metadata, file, indent=2)


## 13. Summarize and save each run

Scores are proportions from 0 to 1. SD describes scan-selection variability on the same test samples, not uncertainty across new corn populations.

In [ ]:
def summarize(records):
    runs = pd.DataFrame(records)
    rows = []
    for k, group in runs.groupby('scan_count', sort=True):
        rows.append(dict(
            scan_count=int(k), repetitions=len(group),
            mean_test_accuracy=group.test_accuracy.mean(),
            sd_test_accuracy=group.test_accuracy.std(ddof=1),
            mean_test_macro_f1=group.test_macro_f1.mean(),
            sd_test_macro_f1=group.test_macro_f1.std(ddof=1),
            mean_measurement_time_s=np.nan,
        ))
    return pd.DataFrame(rows)


In [ ]:
def save_run(result):
    metrics, predictions, selections, model = result
    first_run = len(records) == 0
    records.append(metrics)

    files = [
        ('per_run_metrics.csv', [metrics]),
        ('test_predictions.csv', predictions),
        ('scan_selections.csv', selections)
    ]

    for filename, rows in files:
        pd.DataFrame(rows).to_csv(
            output_folder / filename,
            index=False,
            mode='w' if first_run else 'a',
            header=first_run
        )

    summarize(records).to_csv(
        output_folder / 'ablation_summary.csv', index=False, na_rep='N/A'
    )


## 14. Run the reduced scan counts

30 repetitions each for 1, 5, 10, 15, and 20 scans. The 32-scan baseline is included once. If interrupted, saved files contain partial results; rerun from the top in a new folder if needed.

In [ ]:
require(len(records) == 0, 'Records already exist. Do not run this cell twice.')
save_run(baseline)

for k in COUNTS[:-1]:
    for repetition in range(1, REPETITIONS + 1):
        result = evaluate(df_raw, k, repetition)
        save_run(result)
        scores = result[0]

        print(
            f"Scans: {k:2d} | Run: {repetition:2d}/{REPETITIONS} | "
            f"Accuracy: {scores['test_accuracy']:.4f} | "
            f"Macro-F1: {scores['test_macro_f1']:.4f}"
        )

metadata['status'] = 'completed'
metadata['completed_runs'] = len(records)
metadata['completed_utc'] = datetime.now(timezone.utc).isoformat()
with open(output_folder / 'run_metadata.json', 'w') as file:
    json.dump(metadata, file, indent=2)


## 15. View the summary

Measurement time stays blank here because it comes from the separate physical trials. No timing estimate is calculated from computer runtime.

In [ ]:
df_summary = summarize(records)
print('Results folder:', output_folder.resolve())
df_summary
